# Supply Chain Disruption & Cost Model Training

This notebook trains the ML artifacts used by the Supply Chain GenAI Agent. It creates the XGBoost disruption classifier, shipping-cost regressors, feature configuration, evaluation reports, and mitigation playbook used later by the RAG/LangGraph inference agent.

It creates these downloadable artifacts in Kaggle's `/kaggle/working` directory:

- `supply_chain_agent_training_outputs/models/disruption_classifier.pkl`
- `supply_chain_agent_training_outputs/models/cost_predictor.pkl`
- `supply_chain_agent_training_outputs/models/cost_predictor_enhanced.pkl` (extra improved cost model)
- `supply_chain_agent_training_outputs/knowledge_base/mitigation_playbook.txt`
- `supply_chain_agent_training_outputs/reports/metrics.json`
- `supply_chain_model_artifacts.zip`

No OpenAI key, LangGraph, FAISS, or LangChain setup is needed for this training notebook. It only trains the local ML models and creates the text knowledge base. Build the FAISS index separately with the GenAI agent: `python supply_chain_genai_agent_patched.py index`.

## Kaggle setup

1. Create a Kaggle Notebook.
2. Add your CSV as an input dataset.
3. Import or upload this `.ipynb` file.
4. Run all cells.
5. Download `supply_chain_model_artifacts.zip` from the right-side Output panel.

In [1]:
# Cell 1: Imports and runtime configuration

import os

# Keep XGBoost stable in hosted notebook environments.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

import json
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_JOBS = min(os.cpu_count() or 1, 4)
print(f"Using N_JOBS={N_JOBS}")

Using N_JOBS=4


In [2]:
# Cell 2: Locate the input CSV and configure output paths

CSV_FILENAME = "global_supply_chain_disruption_v1.csv"

search_roots = []
for candidate in [Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]:
    if candidate.exists():
        search_roots.append(candidate)

csv_matches = []
for root in search_roots:
    csv_matches.extend(root.rglob(CSV_FILENAME))

# Fallback: if the exact file name is different in Kaggle, use the first CSV found.
if not csv_matches:
    for root in search_roots:
        csv_matches.extend(root.rglob("*.csv"))

if not csv_matches:
    raise FileNotFoundError(
        "Could not find a CSV file. In Kaggle, attach the dataset containing "
        f"{CSV_FILENAME}, then rerun the notebook."
    )

DATA_PATH = csv_matches[0]
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_DIR = WORKING_DIR / "supply_chain_agent_training_outputs"
MODELS_DIR = OUTPUT_DIR / "models"
KB_DIR = OUTPUT_DIR / "knowledge_base"
REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [OUTPUT_DIR, MODELS_DIR, KB_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Data path : {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Data path : /kaggle/input/datasets/parthkharbanda/global-supply-chain/global_supply_chain_disruption_v1.csv
Output dir: /kaggle/working/supply_chain_agent_training_outputs


In [3]:
# Cell 3: Domain constants, feature lists, and training switches

# This is the operational threshold used by the agent code.
DISRUPTION_PROB_THRESHOLD = 0.35

# Production-safe default: do not train the classifier on post-event fields such as
# Delay_Days-derived delay_ratio or dataset-level cost_percentile. Set this to False
# only for a retrospective demo, never for real-time early-warning inference.
USE_LEAKAGE_SAFE_CLASSIFIER = True

# Keep True to also save an improved cost model beside the agent-compatible model.
TRAIN_ENHANCED_COST_MODEL = True

ROUTE_RISK_MAP = {
    "Suez": 0.91,
    "Pacific": 0.74,
    "Atlantic": 0.58,
    "Intra-Asia": 0.45,
    "Commodity": 0.28,
}

PRODUCT_CRITICALITY_MAP = {
    "Pharmaceuticals": 1.00,
    "Perishable Foods": 0.95,
    "Semiconductors": 0.70,
    "Consumer Electronics": 0.55,
    "Auto Parts": 0.50,
    "Raw Materials": 0.25,
    "Textiles": 0.15,
}

DISRUPTION_LABEL_MAP = {
    "No Disruption": 0,
    "Port Congestion": 1,
    "Geopolitical Conflict (Route Diversion)": 2,
    "Severe Weather (Typhoon/Storm)": 3,
}

NO_DISRUPTION_VALUES = {
    "",
    "none",
    "nan",
    "no disruption",
    "no_disruption",
    "no-disruption",
    "null",
}

CLASSIFIER_FEATURES_DEMO = [
    # Continuous
    "Geopolitical_Risk_Index", "Weather_Severity_Index", "Inflation_Rate_Pct",
    "Shipping_Cost_USD", "Scheduled_Lead_Time_Days",
    # Engineered
    "lead_time_buffer", "delay_ratio", "composite_risk_score",
    "geo_weather_interaction", "route_risk_score", "product_criticality",
    "log_shipping_cost", "cost_percentile",
    # One-hot
    "mode_Sea", "mode_Air",
    "route_Suez", "route_Pacific", "route_Atlantic",
    "route_Intra-Asia", "route_Commodity",
    "product_Pharmaceuticals", "product_Perishable Foods",
    "product_Semiconductors", "product_Auto Parts",
]

# A leakage-safe version for early-warning experimentation.
# It removes final-outcome or likely post-event cost features.
CLASSIFIER_FEATURES_LEAKAGE_SAFE = [
    feature for feature in CLASSIFIER_FEATURES_DEMO
    if feature not in {"delay_ratio", "Shipping_Cost_USD", "log_shipping_cost", "cost_percentile"}
]

CLASSIFIER_FEATURES = (
    CLASSIFIER_FEATURES_LEAKAGE_SAFE
    if USE_LEAKAGE_SAFE_CLASSIFIER
    else CLASSIFIER_FEATURES_DEMO
)

# This list matches the current agent script's build_cost_features() behavior.
COST_MODEL_FEATURES_AGENT_COMPATIBLE = [
    "Geopolitical_Risk_Index", "Weather_Severity_Index",
    "Scheduled_Lead_Time_Days", "route_risk_score", "product_criticality",
    "mode_Sea", "mode_Air", "geo_weather_interaction",
    "route_Suez", "route_Pacific", "route_Atlantic",
]

# Extra improved cost feature list. Use this only if your inference code uses the same feature order.
COST_MODEL_FEATURES_ENHANCED = [
    "Geopolitical_Risk_Index",
    "Weather_Severity_Index",
    "Scheduled_Lead_Time_Days",
    "Base_Lead_Time_Days",
    "lead_time_buffer",
    "Order_Weight_Kg",
    "route_risk_score",
    "product_criticality",
    "geo_weather_interaction",
    "mode_Sea",
    "mode_Air",
    "route_Suez",
    "route_Pacific",
    "route_Atlantic",
    "route_Intra-Asia",
    "route_Commodity",
    "product_Pharmaceuticals",
    "product_Perishable Foods",
    "product_Semiconductors",
    "product_Auto Parts",
    "product_Consumer Electronics",
    "product_Raw Materials",
    "product_Textiles",
]

ALL_EXPECTED_FEATURES = sorted(
    set(CLASSIFIER_FEATURES_DEMO)
    | set(CLASSIFIER_FEATURES_LEAKAGE_SAFE)
    | set(COST_MODEL_FEATURES_AGENT_COMPATIBLE)
    | set(COST_MODEL_FEATURES_ENHANCED)
)

print(f"Classifier mode: {'leakage-safe' if USE_LEAKAGE_SAFE_CLASSIFIER else 'agent-compatible demo'}")
print(f"Classifier feature count: {len(CLASSIFIER_FEATURES)}")
print(f"Agent-compatible cost feature count: {len(COST_MODEL_FEATURES_AGENT_COMPATIBLE)}")
print(f"Enhanced cost feature count: {len(COST_MODEL_FEATURES_ENHANCED)}")

Classifier mode: leakage-safe
Classifier feature count: 20
Agent-compatible cost feature count: 11
Enhanced cost feature count: 23


In [4]:
# Cell 4: Data preparation and feature engineering

def normalize_disruption_event(value: Any) -> Any:
    """Normalize disruption labels and convert placeholder values to NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text.lower() in NO_DISRUPTION_VALUES:
        return np.nan
    canonical = {key.lower(): key for key in DISRUPTION_LABEL_MAP}
    return canonical.get(text.lower(), text)


def ensure_feature_columns(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Ensure every expected one-hot or engineered feature exists."""
    df = df.copy()
    for col in features:
        if col not in df.columns:
            df[col] = 0
    return df


def validate_raw_columns(df: pd.DataFrame) -> None:
    required = {
        "Order_ID",
        "Route_Type",
        "Transportation_Mode",
        "Product_Category",
        "Base_Lead_Time_Days",
        "Scheduled_Lead_Time_Days",
        "Delay_Days",
        "Disruption_Event",
        "Geopolitical_Risk_Index",
        "Weather_Severity_Index",
        "Inflation_Rate_Pct",
        "Shipping_Cost_USD",
        "Order_Weight_Kg",
        "Mitigation_Action_Taken",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Input CSV is missing required columns: {missing}")


def load_and_engineer(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    validate_raw_columns(df)

    df["Disruption_Event"] = df["Disruption_Event"].apply(normalize_disruption_event)
    df["is_disrupted"] = df["Disruption_Event"].notna().astype(int)
    df["disruption_label"] = (
        df["Disruption_Event"]
        .fillna("No Disruption")
        .map(DISRUPTION_LABEL_MAP)
        .fillna(-1)
        .astype(int)
    )
    unknown_labels = sorted(
        df.loc[df["disruption_label"] == -1, "Disruption_Event"].dropna().astype(str).unique()
    )
    if unknown_labels:
        raise ValueError(
            "Unknown Disruption_Event labels found. Add them to DISRUPTION_LABEL_MAP "
            f"or normalize them before training: {unknown_labels}"
        )
    df["is_late"] = (df["Delay_Days"] > 0).astype(int)

    df["lead_time_buffer"] = df["Scheduled_Lead_Time_Days"] - df["Base_Lead_Time_Days"]
    df["delay_ratio"] = df["Delay_Days"] / (df["Scheduled_Lead_Time_Days"] + 1e-6)
    df["geo_weather_interaction"] = df["Geopolitical_Risk_Index"] * df["Weather_Severity_Index"]
    df["composite_risk_score"] = (
        df["Geopolitical_Risk_Index"] * 0.50
        + (df["Weather_Severity_Index"] / 10) * 0.30
        + (df["Inflation_Rate_Pct"] / 10) * 0.20
    )
    df["route_risk_score"] = df["Route_Type"].map(ROUTE_RISK_MAP).fillna(0.5)
    df["product_criticality"] = df["Product_Category"].map(PRODUCT_CRITICALITY_MAP).fillna(0.5)
    df["log_shipping_cost"] = np.log1p(df["Shipping_Cost_USD"])
    df["cost_percentile"] = df.groupby("Route_Type")["Shipping_Cost_USD"].rank(pct=True)
    df["air_viable"] = ((df["product_criticality"] >= 0.5) & (df["Delay_Days"] >= 5)).astype(int)

    for col, prefix in [
        ("Transportation_Mode", "mode"),
        ("Route_Type", "route"),
        ("Product_Category", "product"),
    ]:
        dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int)
        df = pd.concat([df, dummies], axis=1)

    df = ensure_feature_columns(df, ALL_EXPECTED_FEATURES)
    return df

In [5]:
# Cell 5: Load and inspect the dataset

df = load_and_engineer(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns after feature engineering: {df.shape[1]:,}")
print(f"Disruption rate: {df['is_disrupted'].mean():.2%}")
print("\nDisruption_Event distribution:")
display(df["Disruption_Event"].fillna("No Disruption").value_counts(dropna=False).to_frame("count"))

print("\nRoute distribution:")
display(df["Route_Type"].value_counts().to_frame("count"))

print("\nMissing values in active feature sets:")
missing_summary = {
    "classifier_features": int(df[CLASSIFIER_FEATURES].isna().sum().sum()),
    "agent_cost_features": int(df[COST_MODEL_FEATURES_AGENT_COMPATIBLE].isna().sum().sum()),
    "enhanced_cost_features": int(df[COST_MODEL_FEATURES_ENHANCED].isna().sum().sum()),
}
display(pd.DataFrame.from_dict(missing_summary, orient="index", columns=["missing_cells"]))

Rows: 10,000
Columns after feature engineering: 45
Disruption rate: 12.67%

Disruption_Event distribution:


,count
Disruption_Event,
No Disruption,8733
Port Congestion,573
Geopolitical Conflict (Route Diversion),521
Severe Weather (Typhoon/Storm),173



Route distribution:


,count
Route_Type,
Suez,3412
Atlantic,1701
Pacific,1645
Intra-Asia,1634
Commodity,1608



Missing values in active feature sets:


,missing_cells
classifier_features,0
agent_cost_features,0
enhanced_cost_features,0


In [6]:
# Cell 6: Training helpers

def classification_metrics_dict(y_true, y_prob, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_disruption_classifier(data: pd.DataFrame, features: list[str]) -> tuple[XGBClassifier, dict]:
    X = data[features].copy()
    y = data["is_disrupted"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=float(scale_pos),
        subsample=0.80,
        colsample_bytree=0.80,
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = classification_metrics_dict(y_test, y_prob, DISRUPTION_PROB_THRESHOLD)
    metrics["feature_count"] = len(features)
    metrics["mode"] = "leakage_safe" if USE_LEAKAGE_SAFE_CLASSIFIER else "agent_compatible_demo"

    print("Disruption classifier metrics at agent threshold:")
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
    print("\nClassification report at threshold", DISRUPTION_PROB_THRESHOLD)
    print(classification_report(y_test, (y_prob >= DISRUPTION_PROB_THRESHOLD).astype(int), zero_division=0))

    importance = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    importance.to_csv(REPORTS_DIR / "classifier_feature_importance.csv", index=False)
    print("Top classifier features:")
    display(importance.head(15))

    joblib.dump(model, MODELS_DIR / "disruption_classifier.pkl")
    return model, metrics


def train_cost_predictor(
    data: pd.DataFrame,
    features: list[str],
    output_name: str,
    report_prefix: str,
) -> tuple[XGBRegressor, dict]:
    X = data[features].copy()
    y = data["Shipping_Cost_USD"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    model = XGBRegressor(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.80,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)
    abs_error = np.abs(y_test - y_pred)
    pct_error = abs_error / np.maximum(np.abs(y_test), 1e-6)

    metrics = {
        "model_file": output_name,
        "feature_count": len(features),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "median_absolute_error": float(np.median(abs_error)),
        "mape_pct": float(mean_absolute_percentage_error(y_test, y_pred) * 100),
        "median_ape_pct": float(np.median(pct_error) * 100),
        "r2": float(r2_score(y_test, y_pred)),
    }

    print(f"{report_prefix} cost predictor metrics:")
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

    importance = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    importance.to_csv(REPORTS_DIR / f"{report_prefix.lower()}_cost_feature_importance.csv", index=False)
    print(f"Top {report_prefix} cost features:")
    display(importance.head(15))

    joblib.dump(model, MODELS_DIR / output_name)
    return model, metrics

In [7]:
# Cell 7: Train and save the models

classifier_model, classifier_metrics = train_disruption_classifier(df, CLASSIFIER_FEATURES)

cost_model, cost_metrics = train_cost_predictor(
    df,
    COST_MODEL_FEATURES_AGENT_COMPATIBLE,
    output_name="cost_predictor.pkl",
    report_prefix="agent_compatible",
)

all_metrics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_path": str(DATA_PATH),
    "row_count": int(len(df)),
    "use_leakage_safe_classifier": bool(USE_LEAKAGE_SAFE_CLASSIFIER),
    "disruption_classifier": classifier_metrics,
    "cost_predictor_agent_compatible": cost_metrics,
}

if TRAIN_ENHANCED_COST_MODEL:
    enhanced_cost_model, enhanced_cost_metrics = train_cost_predictor(
        df,
        COST_MODEL_FEATURES_ENHANCED,
        output_name="cost_predictor_enhanced.pkl",
        report_prefix="enhanced",
    )
    all_metrics["cost_predictor_enhanced"] = enhanced_cost_metrics

Disruption classifier metrics at agent threshold:


,value
threshold,0.35
roc_auc,0.516266
pr_auc,0.131034
accuracy,0.5445
precision,0.139254
recall,0.501976
f1,0.218026
tn,962
fp,785
fn,126



Classification report at threshold 0.35
              precision    recall  f1-score   support

           0       0.88      0.55      0.68      1747
           1       0.14      0.50      0.22       253

    accuracy                           0.54      2000
   macro avg       0.51      0.53      0.45      2000
weighted avg       0.79      0.54      0.62      2000

Top classifier features:


,feature,importance
11,route_Suez,0.064211
9,mode_Sea,0.056076
6,geo_weather_interaction,0.056046
13,route_Atlantic,0.056027
15,route_Commodity,0.055821
5,composite_risk_score,0.054493
14,route_Intra-Asia,0.053837
2,Inflation_Rate_Pct,0.052946
1,Weather_Severity_Index,0.052293
17,product_Perishable Foods,0.052100


agent_compatible cost predictor metrics:


...

,feature,importance
5,mode_Sea,0.558783
2,Scheduled_Lead_Time_Days,0.228117
6,mode_Air,0.040531
8,route_Suez,0.029507
0,Geopolitical_Risk_Index,0.025325
3,route_risk_score,0.025300
7,geo_weather_interaction,0.023454
1,Weather_Severity_Index,0.020823
9,route_Pacific,0.020803
10,route_Atlantic,0.017467


enhanced cost predictor metrics:


,value
model_file,cost_predictor_enhanced.pkl
feature_count,23
mae,2065.941398
median_absolute_error,221.18998
mape_pct,13.707093
median_ape_pct,6.441035
r2,0.891734


Top enhanced cost features:


,feature,importance
9,mode_Sea,0.625052
2,Scheduled_Lead_Time_Days,0.220126
5,Order_Weight_Kg,0.049557
3,Base_Lead_Time_Days,0.025934
15,route_Commodity,0.011770
0,Geopolitical_Risk_Index,0.009391
6,route_risk_score,0.007852
11,route_Suez,0.007671
4,lead_time_buffer,0.007380
1,Weather_Severity_Index,0.006693


In [8]:
# Cell 8: Build the text knowledge base used by the RAG layer

def build_knowledge_base(data: pd.DataFrame) -> list[str]:
    docs = []

    disrupted = data[data["is_disrupted"] == 1].copy()
    if not disrupted.empty:
        stats = (
            disrupted
            .groupby(["Disruption_Event", "Mitigation_Action_Taken"], dropna=False)
            .agg(
                count=("Order_ID", "count"),
                avg_delay=("Delay_Days", "mean"),
                avg_cost=("Shipping_Cost_USD", "mean"),
                on_time_rate=("is_late", lambda x: 1 - x.mean()),
            )
            .reset_index()
            .sort_values(["Disruption_Event", "on_time_rate"], ascending=[True, False])
        )
        for _, row in stats.iterrows():
            docs.append(
                f"MITIGATION PLAYBOOK RECORD\n"
                f"Disruption Type      : {row['Disruption_Event']}\n"
                f"Mitigation Action    : {row['Mitigation_Action_Taken']}\n"
                f"Historical On-Time   : {row['on_time_rate']:.1%}\n"
                f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
                f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}\n"
                f"Sample Size          : {int(row['count'])} orders"
            )

    route_stats = data.groupby("Route_Type").agg(
        avg_geo_risk=("Geopolitical_Risk_Index", "mean"),
        avg_weather=("Weather_Severity_Index", "mean"),
        disruption_rate=("is_disrupted", "mean"),
        avg_delay=("Delay_Days", "mean"),
        avg_cost=("Shipping_Cost_USD", "mean"),
        order_count=("Order_ID", "count"),
    ).reset_index()
    for _, row in route_stats.iterrows():
        docs.append(
            f"ROUTE RISK PROFILE\n"
            f"Route                : {row['Route_Type']}\n"
            f"Avg Geopolitical Risk: {row['avg_geo_risk']:.3f}\n"
            f"Avg Weather Severity : {row['avg_weather']:.2f}\n"
            f"Disruption Rate      : {row['disruption_rate']:.1%}\n"
            f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
            f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}\n"
            f"Total Orders         : {int(row['order_count'])}"
        )

    product_stats = data.groupby("Product_Category").agg(
        disruption_rate=("is_disrupted", "mean"),
        avg_delay=("Delay_Days", "mean"),
        avg_cost=("Shipping_Cost_USD", "mean"),
        criticality=("product_criticality", "first"),
    ).reset_index()
    for _, row in product_stats.iterrows():
        docs.append(
            f"PRODUCT RISK PROFILE\n"
            f"Product Category     : {row['Product_Category']}\n"
            f"Criticality Score    : {row['criticality']:.2f}\n"
            f"Disruption Rate      : {row['disruption_rate']:.1%}\n"
            f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
            f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}"
        )

    playbook_path = KB_DIR / "mitigation_playbook.txt"
    playbook_path.write_text("\n\n---\n\n".join(docs), encoding="utf-8")
    return docs

kb_docs = build_knowledge_base(df)
print(f"Knowledge-base documents written: {len(kb_docs)}")
print(f"Path: {KB_DIR / 'mitigation_playbook.txt'}")

Knowledge-base documents written: 18
Path: /kaggle/working/supply_chain_agent_training_outputs/knowledge_base/mitigation_playbook.txt


In [9]:
# Cell 9: Save feature configuration and metrics

def _cost_reference_stats(frame: pd.DataFrame) -> dict[str, float]:
    """Return JSON-serializable cost summary/quantiles for inference-time percentile estimates."""
    series = frame["Shipping_Cost_USD"].dropna().astype(float)
    if series.empty:
        return {}
    quantiles = series.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    return {
        "min": float(series.min()),
        "p10": float(quantiles.loc[0.10]),
        "p25": float(quantiles.loc[0.25]),
        "p50": float(quantiles.loc[0.50]),
        "p75": float(quantiles.loc[0.75]),
        "p90": float(quantiles.loc[0.90]),
        "max": float(series.max()),
        "mean": float(series.mean()),
        "std": float(series.std(ddof=0)),
        "count": int(series.shape[0]),
    }

route_cost_reference_stats = {
    str(route): _cost_reference_stats(group)
    for route, group in df.groupby("Route_Type", dropna=False)
}
route_cost_reference_stats["__global__"] = _cost_reference_stats(df)

feature_config = {
    "disruption_probability_threshold": DISRUPTION_PROB_THRESHOLD,
    "use_leakage_safe_classifier": USE_LEAKAGE_SAFE_CLASSIFIER,
    "classifier_features_active": CLASSIFIER_FEATURES,
    "classifier_features_demo_agent_compatible": CLASSIFIER_FEATURES_DEMO,
    "classifier_features_leakage_safe": CLASSIFIER_FEATURES_LEAKAGE_SAFE,
    "cost_model_features_agent_compatible": COST_MODEL_FEATURES_AGENT_COMPATIBLE,
    "cost_model_features_enhanced": COST_MODEL_FEATURES_ENHANCED,
    "route_risk_map": ROUTE_RISK_MAP,
    "product_criticality_map": PRODUCT_CRITICALITY_MAP,
    "disruption_label_map": DISRUPTION_LABEL_MAP,
    "route_cost_reference_stats": route_cost_reference_stats,
}

(MODELS_DIR / "feature_config.json").write_text(
    json.dumps(feature_config, indent=2),
    encoding="utf-8",
)
(REPORTS_DIR / "metrics.json").write_text(
    json.dumps(all_metrics, indent=2),
    encoding="utf-8",
)

print("Saved:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(OUTPUT_DIR))


Saved:
- knowledge_base/mitigation_playbook.txt
- models/cost_predictor.pkl
- models/cost_predictor_enhanced.pkl
- models/disruption_classifier.pkl
- models/feature_config.json
- reports/agent_compatible_cost_feature_importance.csv
- reports/classifier_feature_importance.csv
- reports/enhanced_cost_feature_importance.csv
- reports/metrics.json


In [10]:
# Cell 10: Smoke test the saved model files

loaded_classifier = joblib.load(MODELS_DIR / "disruption_classifier.pkl")
loaded_cost_model = joblib.load(MODELS_DIR / "cost_predictor.pkl")

sample = df.iloc[0]
classifier_vector = sample[CLASSIFIER_FEATURES].astype(float).to_numpy().reshape(1, -1)
cost_vector = sample[COST_MODEL_FEATURES_AGENT_COMPATIBLE].astype(float).to_numpy().reshape(1, -1)

prob = float(loaded_classifier.predict_proba(classifier_vector)[0, 1])
cost = float(loaded_cost_model.predict(cost_vector)[0])

print(f"Sample Order_ID: {sample['Order_ID']}")
print(f"Predicted disruption probability: {prob:.4f}")
print(f"Predicted shipping cost: ${cost:,.2f}")

if TRAIN_ENHANCED_COST_MODEL:
    loaded_enhanced_cost_model = joblib.load(MODELS_DIR / "cost_predictor_enhanced.pkl")
    enhanced_vector = sample[COST_MODEL_FEATURES_ENHANCED].astype(float).to_numpy().reshape(1, -1)
    enhanced_cost = float(loaded_enhanced_cost_model.predict(enhanced_vector)[0])
    print(f"Enhanced cost model prediction: ${enhanced_cost:,.2f}")

Sample Order_ID: ORD-00BCB25B
Predicted disruption probability: 0.2871
Predicted shipping cost: $4,261.71
Enhanced cost model prediction: $1,474.45


In [11]:
# Cell 11: Create one downloadable ZIP file

zip_base = WORKING_DIR / "supply_chain_model_artifacts"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)

print("Created downloadable artifact:")
print(zip_path)
print("\nIn Kaggle, download this ZIP from the notebook Output panel.")

Created downloadable artifact:
/kaggle/working/supply_chain_model_artifacts.zip

In Kaggle, download this ZIP from the notebook Output panel.


## FAISS index note

This notebook intentionally does **not** build the FAISS vector index. It only creates `knowledge_base/mitigation_playbook.txt`. After downloading the artifacts and placing the GenAI agent beside them, build the RAG index with:

```bash
python supply_chain_genai_agent_patched.py index
```

That command creates:

```text
supply_chain_agent_training_outputs/knowledge_base/faiss_index/index.faiss
supply_chain_agent_training_outputs/knowledge_base/faiss_index/index.pkl
```

## Using the generated files with your patched GenAI agent

After running this notebook, download `supply_chain_model_artifacts.zip` and extract it beside the GenAI agent file:

```text
project_folder/
  supply_chain_genai_agent_patched.py
  supply_chain_agent_training_outputs/
    models/
      disruption_classifier.pkl
      cost_predictor.pkl
      cost_predictor_enhanced.pkl
      feature_config.json
    knowledge_base/
      mitigation_playbook.txt
```

Then run:

```bash
python supply_chain_genai_agent_patched.py check
python supply_chain_genai_agent_patched.py index
```

Use `--replay-mode` only when replaying historical CSV rows where `Disruption_Event` should be treated as a known alert:

```bash
python supply_chain_genai_agent_patched.py resolve --order-idx 10 --replay-mode
```

For real-time JSON inputs, omit `--replay-mode` so `Disruption_Event` cannot bypass the classifier.